# Tests de l'API FastAPI

Ce notebook vérifie les deux endpoints attendus de l'API :

- `POST /predict`
- `POST /recommend`

Prérequis : lancer l'API avant d'exécuter ce notebook.

```bash
uvicorn main:app --reload
```


transformer le dataset historique en dataset non-historique
chaque pays -> colonne date de rendement (pouvoir donner min, max, std)

expérience 1
xgboost et lightgbm à fournir (laisser exprimer les paramètres hors temporelle)

expérience 2 données simulées
voir si hishtgbm / sarimax permet de gagner du temps sur le feature engeeniring

xgboost-random forset

ex

In [24]:
print(2**15)

32768


In [ ]:
import json
from pathlib import Path
from urllib import error, request
import pandas as pd

from scripts.project_config import load_preparation_config

test_config = load_preparation_config()
DATASET_PATH = test_config["DATASET_PATH"]
dataset = pd.read_csv(DATASET_PATH)
sample_row = dataset.dropna(subset=["average_rain_fall_mm_per_year", "avg_temp"]).iloc[0]

print(f"Dataset utilisé : {DATASET_PATH}")


## Payloads

Les payloads de test reflètent maintenant la logique produit retenue :

- l'utilisateur renseigne sa surface en hectares ;
- l'année n'est plus envoyée au client ;
- l'API injecte automatiquement l'année courante au moment de la prédiction.


  ### POST /predict
  Prend en entrée les données d'une seule culture et son contexte, et retourne la prédiction de rendement.

In [17]:
predict_payload = {
    "area": "France",
    "crop": "Maize",
    "hectares": 12.5,
    "average_rain_fall_mm_per_year": 327.0,
    "pesticides_tonnes": None,
    "avg_temp": 15.0,
}


### POST /recommend
Prend en entrée uniquement le contexte (température, etc.), et retourne une liste de toutes les cultures possibles, triées par rendement prédit décroissant.

In [18]:
recommend_payload = {
    "area": "Spain",
    "hectares": 12.5,
    "average_rain_fall_mm_per_year": 327.0,
    "pesticides_tonnes": None,
    "avg_temp": 20.0,
    # Optionnel : restreindre la recommandation à certaines cultures
    # "candidate_crops": ["Maize", "Wheat", "Rice, paddy"],
}


## Fonction utilitaire HTTP

Cette fonction envoie une requête JSON à l'API et renvoie le code HTTP et le corps de réponse.


In [19]:
def post_json(endpoint, payload):
    body = json.dumps(payload).encode("utf-8")
    req = request.Request(
        f"http://127.0.0.1:8000{endpoint}",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with request.urlopen(req) as response:
            response_body = response.read().decode("utf-8")
            return response.status, json.loads(response_body)
    except error.URLError as exc:
        raise RuntimeError(
            "Impossible de joindre l'API. Lancer `uvicorn main:app --reload` avant d'exécuter ce notebook."
        ) from exc


## Test de `POST /predict`


In [20]:
predict_status, predict_response = post_json("/predict", predict_payload)

assert predict_status == 200
assert "predicted_yield_t_ha" in predict_response
assert "predicted_total_production_tons" in predict_response
assert "year_used" in predict_response
assert predict_response["hectares"] == predict_payload["hectares"]

{
    "payload": predict_payload,
    "response": predict_response,
}


{'payload': {'area': 'France',
  'crop': 'Maize',
  'hectares': 12.5,
  'average_rain_fall_mm_per_year': 327.0,
  'pesticides_tonnes': None,
  'avg_temp': 15.0},
 'response': {'year_used': 2026,
  'hectares': 12.5,
  'predicted_yield_t_ha': 1.9015,
  'predicted_total_production_tons': 23.7688}}

## Test de `POST /recommend`


In [21]:
recommend_status, recommend_response = post_json("/recommend", recommend_payload)
recommendations = recommend_response["recommendations"]

assert recommend_status == 200
assert "year_used" in recommend_response
assert recommend_response["hectares"] == recommend_payload["hectares"]
assert len(recommendations) > 0
assert recommendations == sorted(
    recommendations,
    key=lambda row: row["predicted_total_production_tons"],
    reverse=True,
)

pd.DataFrame(recommendations).head(10)


,crop,predicted_yield_t_ha,predicted_total_production_tons
0,Potatoes,13.9303,174.1293
1,Sweet potatoes,10.0104,125.1303
2,Yams,8.9179,111.4742
3,Cassava,8.7512,109.3900
4,Plantains and others,6.9926,87.4076
5,"Rice, paddy",2.7037,33.7965
6,Maize,1.8378,22.9719
7,Wheat,1.8377,22.9711
8,Soybeans,1.2571,15.7140
9,Sorghum,0.8280,10.3506
